# Intro to Latent Variable Models with Pollux

In [ ]:
import daft
import jax
import matplotlib.pyplot as plt
import numpy as np
import numpyro.distributions as dist

import pollux as plx
from pollux.models.transforms import (
    LinearTransform,
    PolyFeatureTransform,
    TransformSequence,
)

plt.rcParams["figure.constrained_layout.use"] = False
rng = np.random.default_rng(42)

Latent Variable Models (LVMs) are a general class of models that are useful for many types of problems. For the purposes of Pollux and this note, LVMs are generative models of data in which some of the variables are unobserved (i.e. *latent*) and some are noisily observed. More specifically within Pollux, we split the latent parameters of a model into two conceptual classes: we refer to vector-valued, per-object latent variables as _the latents_ and all other (even latent) parameters are the _coefficients_ or parameters of the model. This distinction might seem arbitrary, but the point is that _the latents_ scale with the number of objects in the dataset whereas the _coefficients_ do not. In this context, _the latents_ are often used to capture (or discover) structure in the data and are sometimes interpretable. These types of generative models are related to machine learning methods and concepts like [embeddings](https://en.wikipedia.org/wiki/Embedding_(machine_learning)), [representation learning](https://en.wikipedia.org/wiki/Representation_learning), and [autoencoders](https://en.wikipedia.org/wiki/Variational_autoencoder).

A general [probabilistic graphical model](https://adrian.pw/blog/probabilistic-graphical-models/) (PGM) for a latent variable model of the form used throughout Pollux is shown below:

In [ ]:
pgm = daft.PGM(observed_style="shaded", dpi=150)
pgm.add_node("theta", r"$\theta$", 1, 3)
pgm.add_node("z", r"$z_n$", 2, 2)
pgm.add_node("y", r"$y_n$", 1, 1, observed=True)
for parent, child in [("z", "y"), ("theta", "y")]:
    pgm.add_edge(parent, child)
pgm.add_plate(
    [0.5, 0.5, 2.0, 2.0], label=r"$n = 1 \ldots N$", shift=0.0, position="bottom right"
)
pgm.render();

In the PGM above, $y_n$ are the observed data for object index $n$, $z_n$ are the latent variables for object index $n$, and $\theta$ are the coefficients of the model. (Remember that PGMs do not show the functional form of the model, only the conditional dependencies between the variables, so we still haven't specified how $\theta$ and $z_n$ combine to produce $y_n$.) The plate indicates that there are $N$ objects in the dataset, each with its own latent variable(s) $z_n$ and observed data $y_n$. In general, $y$ and $z$ are vector valued and $\theta$ may be an arbitrarily-shaped tensor.

The PGM encodes a factorization of the joint posterior probability of the latents and coefficients given the data. Reading it off directly --- the plate suggests a product over objects, and $\theta$ sits outside the plate, so it appears once

$$
p(\{z_n\}, \theta \mid \{y_n\}) \propto p(\theta) \, 
    \prod_{n=1}^{N} p(y_n \mid z_n, \theta) \, p(z_n)
$$

The three pieces are the likelihood of the data given the latents and coefficients $p(y_n \mid z_n, \theta)$, the prior on the latents $p(z_n)$, and the prior on the coefficients $p(\theta)$. 

This is a very general model structure, and many models you may be familiar with can be thought of as special cases of a general LVM. For example: 
- Principal component analysis (PCA) -- specifically probabilistic PCA -- can be interpreted as an LVM in which the latents are the coefficients of the principal components and $\theta$ are the principal component vectors. (Or, more generally than pPCA, factor analysis is an LVM in which the latents are the coefficients of the factors and $\theta$ are the factors).
- Linear regression can be thought of as (degenerate form of) a LVM in which the latents are the true (noiseless) values of $y_n$ and the coefficients $\theta$ as the vector of linear parameters (e.g., slope and intercept). In this case, at fixed $\theta$ and covariates, each $z_n$ is determined rather than uncertain, so the per-object latents aren't actually free parameters.
- Gaussian mixture models (GMMs) are an example of an LVM where some latents are discrete: in a GMM, each $z_n$ is a discrete assignment of object $n$ to one of $K$ components, whereas the coefficients $\theta$ are the means, covariances, and mixing weights of those components.
- Various data-driven stellar spectroscopy models (e.g., _The Cannon_, _Lux_) are also LVMs. In this case, the data are the observed spectra of stars and the latents are either fixed to (a transformation of) the stellar labels (effective temperature, surface gravity, chemical abundances, etc. in the case of _The Cannon_) or left free (in the case of _Lux_). The coefficients describe how the flux at each wavelength responds to the latents, and are shared across every star in the survey. 

At this point you might be wondering: are all probabilistic models LVMs? The answer is that many (or most) generative models can be written in the form of an LVM, but it is useful to think of a model as an LVM depends on the context. For example, it is not particularly useful to think of standard linear regression model as an LVM, but fitting a linear model with uncertainties on both $x$ and $y$ and with missing or censored data might be useful to structure as an LVM.

Let's look at some more concrete examples.

## A linear model as a simple LVM

The PGM above does not define the functional form of the model or relationships between the latents and outputs (data). For this example, we'll use a linear model to make some of these concepts more concrete. For this model:

$$
\boldsymbol{y} = \boldsymbol{\mathrm{A}} \, \boldsymbol{z} + \mathrm{noise}
$$

where $z$ is a $L=3$ dimensional latent space, $y$ is a $D=128$ dimensional data vector, and therefore the matrix $\mathrm{A}$ is a $(D, L) = (128, 3)$ dimensional mapping from latents to data. For example, imagine that the data $y$ is a spectrum with 128 pixels and the latents represent some intrinsic properties of the sources. Because this is a linear model, the columns of $\mathrm{A}$ are often called basis vectors and the corresponding latent values are the basis weights, but in our terminology $z$ are _the latents_ and the elements of $\mathrm{A}$ are _the coefficients_.

Let's make some simulated data with this structure for $N=512$ objects. We'll put in three basis vectors to the columns of $A$ to make the simulated data more visually interesting (inspired by absorption lines in a stellar spectrum):

In [ ]:
N = 512
D = 128
L = 3

rng = np.random.default_rng(123)

A = np.zeros((D, L))
pix = np.arange(D)
A[:, 0] = np.exp(-0.5 * (pix - 64) ** 2 / 10**2)
A[:, 1] = np.exp(-0.5 * (pix - 32) ** 2 / 2**2)
A[:, 2] = np.exp(-0.5 * (pix - 96) ** 2 / 5**2)

z = -np.abs(rng.normal(size=(N, L)))
true_y = z @ A.T

# heteroskedastic noise
y_err = 10 ** (rng.uniform(-1.0, 0.5, size=(N, D)))

y = true_y + rng.normal(scale=y_err)

Here are the true basis vectors:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(A, marker="")
_ = ax.set(xlabel="pixel", ylabel="basis vector value", title="true basis vectors")

And now some of the simulated data (the first 4 objects) with error bars. Notice that the data are quite noisy:

In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(10, 6), layout="constrained", sharex=True, sharey=True
)
for i in range(4):
    ax = axes.flat[i]
    ax.scatter(pix, y[i], marker="o", s=5, color="k", zorder=10)
    ax.errorbar(
        pix, y[i], yerr=y_err[i], marker="none", color="#aaaaaa", zorder=5, ls="none"
    )
    ax.set(xlim=(0, D), ylim=(-3, 3))
    ax.text(5, 2.8, f"Object {i + 1}", ha="left", va="top", fontsize=14)

If there was no measurement uncertainty, we could recover (a linear combination of) the basis vectors using a standard singular value decomposition (SVD): in the Figure below, the left panel shows the SVD of the true $y$ values, demonstrating that there is a huge drop off in power after 3 components, indicating that the data can be well decomposed into a combination of three basis vectors. The right panel of the Figure shows the SVD of the noisy data --- here there is a more "flat" distribution of singular values because of the heteroskedastic noise. 

In [ ]:
true_svd = np.linalg.svd(true_y, compute_uv=True)
err_svd = np.linalg.svd(y, compute_uv=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained", sharex=True)
names = ["SVD of true $y$", "SVD of observed (noisy) $y$"]
axes[0].plot(
    np.arange(1, 11),
    true_svd.S[:10],
    marker="o",
    ls="none",
)
axes[1].plot(
    np.arange(1, 11),
    err_svd.S[:10],
    marker="o",
    ls="none",
)
for ax, name in zip(axes, names):
    ax.axvline(L + 0.5, color="#aaaaaa", ls="--", lw=1)
    _ = ax.set(
        xlabel="component",
        yscale="log",
        title=name,
        xlim=(0.5, 10.5),
        xticks=np.arange(1, 11),
    )

_ = axes[0].set_ylabel("singular value")

We can also look at the basis vectors that the SVD returns. In the Figure below, the left panel shows the first three right singular vectors of the *true* $y$ values and the right panel shows the same for the noisy data.

Two things appear from this. First, even in the noiseless case, the recovered vectors are not the three input basis vectors: they are orthogonal linear combinations of the input basis vectors, because an SVD returns the directions of greatest variance. Second, in the noisy case they are visibly noisy as well (i.e. noise leaks into the basis vectors) --- the SVD has no way of knowing that some pixels have much larger uncertainties than others, so a pixel with a huge uncertainty will bias the solution.

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(12, 4), layout="constrained", sharex=True, sharey=True
)
names = ["SVD basis vectors for true $y$", "SVD basis vectors for observed (noisy) $y$"]
axes[0].plot(
    pix,
    true_svd.Vh[:3].T,
    marker="",
)
axes[1].plot(
    pix,
    err_svd.Vh[:3].T,
    marker="",
)
for ax, name in zip(axes, names):
    _ = ax.set(
        xlabel="pixel",
        title=name,
    )
_ = axes[0].set_ylabel("basis vector value")

### Limitations with an SVD (i.e. PCA) of noisy data

When data are high signal-to-noise and when you expect the underlying model to be close to linear, an SVD (or PCA) of your data can form a useful decomposition of the data. However, there are a number of limitations that motivate using a linear LVM for data instead:

1. If the uncertainties are heteroskedastic (i.e., differ by object or by element of the data): An SVD minimizes an unweighted squared error, which is only the right thing to do if every dimension of every object has the same noise properties. A spectrum with a bright sky line in one pixel or a survey with a mix of source brightness (and therefore signal-to-noise) violates this.
2. If data have missing values: Not every object is measured in every dimension. If you want to use SVD, you have to either infill missing data or limit your SVD to a subset of the sample with perfect coverage in the data, which is often not a representative sample of your full data set. SVDs have no natural way of handling missing values.
3. If there is more than one kind of observation: In the example above, we are considering a single mapping from latents to spectrum, but we often have other kinds of data that we might want to use simultaneously --- for example, we may have spectra, existing stellar labels, broadband photometry, and astrometry for the same sources. The different data outputs have different dimensionalities and dimensions, but might share a common latent representation.
4. If the relationship between the latents and the data is not linear: This is the case in our toy example, but doesn't have to be (as we explore later).

The first two (1 and 2) of these limitations are addressed by writing down a likelihood instead of a matrix norm, which is the step from PCA to probabilistic PCA. The 3rd point is addressed by letting the likelihood have several output blocks, each with its own function of the latents. The 4th point is addressed by allowing more complex mappings from latents to predicted outputs. Pollux implements and supports all four of these things with the LVM in {py:class}`~pollux.models.LVM`.

For this linear model, with Gaussian uncertainties $\sigma_n$ on each element of $y_n$, the likelihood is

$$
p(y_n \mid z_n, \theta) = \mathcal{N}\left(y_n \mid \mathrm{A}, z_n, \Sigma_n\right)
\quad \mathrm{with} \quad \Sigma_n = \mathrm{diag}(\sigma_n^2)
$$

and finding the MAP parameters (i.e. optimizing the Pollux model) means minimizing the negative log of the expression above,

$$
-\ln p = \sum_{n=1}^{N} \left[ \frac{1}{2} \left(y_n - \mathrm{A} \, z_n\right)^\top \Sigma_n^{-1} \left(y_n - \mathrm{A} \, z_n\right) - \ln p(z_n) \right] -
\ln p(\theta) + \mathrm{const.}
$$

Let's see how we could implement this model for the toy data we generated above using Pollux.

First, we wrap the data in a {py:class}`~pollux.data.PolluxData` instance:

In [ ]:
data = plx.data.PolluxData(
    flux=plx.data.OutputData(y, err=y_err),
)

Then we construct the model and run an iterative optimization scheme to infer the latents and basis vectors (elements of $A$).

In [ ]:
model = plx.LVM(latent_size=L)
model.register_output("flux", LinearTransform(output_size=D))

trained = model.optimize_iterative(
    data, max_cycles=64, rng_key=jax.random.PRNGKey(0), progress=False
)
trained_A = trained.params["flux"]["data"]["A"]

(Under the hood, by default, this assumes unit variance Normal priors on the latents and all elements of $A$ -- we customize these choices below.)

Let's look at the recovered _maximum a posteriori_ (MAP) basis vectors:

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(trained_A, marker="")
_ = ax.set(xlabel="pixel", ylabel="basis vector value", title="inferred basis vectors")

(We could also connect this model to [numpyro](https://num.pyro.ai/) and run MCMC sampling to get posterior samples, but we won't demonstrate that here.)

The inferred basis vectors are much smoother, but since we know the ground truth here, we know that the inferred basis vectors are linear combinations of the true basis vectors. This is a known fact about PCA and probabilistic PCA: we can always rotate the latent vectors ($\mathrm{R}$) and inverse rotate ($\mathrm{R}^\top$) the basis vectors and get back the same mapping to predicted data:

$$
\mathrm{A} \, z_n = \left(\mathrm{A} \, \mathrm{R}^{-1}\right)\left(\mathrm{R} \, z_n\right) 
$$

For our example simulated data, we know that the data are generated such that the basis vectors are always positive and the coefficients are always negative (or vice versa). So instead of PCA or pPCA, we could use non-negative matrix factorization (NMF). NMF is like PCA, but with the additional constraints that:

$$
A_{ij} \geq 0 \\
z_{i} \geq 0
$$

for all elements of $A$ and $z$ (in our case, $z$ should be negative, and that is easy to handle in standard NMF).

In Pollux, we can enforce these constraints by using bounded priors on the elements of $\mathrm{A}$ and the latents $z$. Here we will create a second version of the model that uses unit variance Gaussians for both, but truncated to either be positive (for $A$) or to be negative (for $z$):

In [ ]:
model_nmf = plx.LVM(latent_size=L)
latents_prior = dist.TruncatedNormal(scale=1.0, high=0.0)
model_nmf.register_output(
    "flux", LinearTransform(output_size=D, priors={"A": dist.HalfNormal(1.0)})
)

Now we optimize the model to get MAP values of $A$ and $z$ for this non-negative model:

In [ ]:
trained_nmf = model_nmf.optimize_iterative(
    data,
    max_cycles=256,
    rng_key=jax.random.PRNGKey(0),
    progress=False,
    latents_prior=latents_prior,
)
trained_nmf_A = trained_nmf.params["flux"]["data"]["A"]

This time the inferred basis vectors look like the ones we put in: three roughly
Gaussian features in the right places, rather than mixtures of all three.

The reason is that the non-negativity constraint removes the rotational freedom we
discussed above. A rotation $\mathrm{R}$ that mixes the basis vectors will generally
produce some negative entries in $\mathrm{A} \, \mathrm{R}^{-1}$ or in
$\mathrm{R} \, z_n$, and those solutions are now excluded by the priors, so the
optimizer cannot drift along the rotation and has to settle on the one combination that
keeps everything signed consistently.

Here, we recovered the input basis vectors because we happened to know how the data were
generated and could impose a constraint that was actually true to the way the data are generated. Non-negativity is a modeling assumption: if the real components had some negative values, this model would be wrong, and it would still return a confident-looking (wrong) answer. In this toy case, it also helps that our three features barely overlap, so that some pixels are dominated by a
single component --- NMF is much less likely to uniquely determine the components when they are heavily blended.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(trained_nmf_A, marker="")
_ = ax.set(xlabel="pixel", ylabel="basis vector value", title="inferred basis vectors")

## Latent Variable Models can be nonlinear

Nothing in the text above or the PGM requires that $y_n$ be a *linear* function
of $z_n$; that was a choice we made to keep the first example simple. In general the
mapping can be any function, and it is worth noting two different places that
nonlinearity can enter into LVMs:

1. The predicted data can be nonlinear in **the latents**. _The Cannon_ ([Ness et al. 2015](https://arxiv.org/abs/1501.07604)) is an example of such a model in stellar spectroscopy: the latents are the stellar labels and the predicted spectrum is a quadratic function of the labels. A Gaussian process latent variable model (GPLVM) is another example where the mapping is nonparametric --- see [Eilers et al. 2022](https://arxiv.org/abs/2209.02725) for an application of a GPLVM to quasar spectra.
2. The predicted data can be nonlinear in **the coefficients**: A model could pass the coefficients through a nonlinear function. For example, we could have used a `softplus` to force the basis vectors to be non-negative, rather than imposing this constraint through a prior as we did above.

This distinction matters because it affects how we might optimize the model. If the model is linear in either the latents or the coefficients, then we can use a linear solver to find the optimal values of that block of parameters while holding the other block fixed. This is what
{py:meth}`~pollux.models.LVM.optimize_iterative` does: it alternates between optimizing the latents and the coefficients, using a linear solver for each block. If the model is nonlinear in both, then we have to use a nonlinear solver for all blocks, which is slower and more prone to getting stuck in local minima. The default optimization scheme for such models uses the Adam optimizer through the {py:meth}`~pollux.models.LVM.optimize` method.

In the next example, we'll build a model in which the flux is a quadratic function of the latents. In Pollux, we can use a {py:class}`~pollux.models.transforms.TransformSequence` for this: we first expand the latents into polynomial features, then map those to the data with a linear layer.

In [ ]:
quadratic_output = TransformSequence(
    (PolyFeatureTransform(degree=2), LinearTransform(output_size=D))
)

model_nonlinear = plx.LVM(latent_size=L)
model_nonlinear.register_output("flux", quadratic_output)

# the coefficient matrix is now (D, n_features) rather than (D, L): a quadratic in
# L=3 latents has 10 features -- a constant, the 3 latents, and their 6 products
coeff_prior = quadratic_output.get_expanded_priors(latent_size=L)["1:A"]
coeff_prior.batch_shape

In [ ]:
trained_nonlinear = model_nonlinear.optimize_iterative(
    data, max_cycles=128, rng_key=jax.random.PRNGKey(0), progress=False
)

# how each block was actually solved
[(block.name, block.optimizer) for block in trained_nonlinear.blocks]

Note the warning, and the block summary above: the coefficients are still solved exactly (using `least_square`, which is a linear solver), because the predicted flux is linear in them once the polynomial features are computed. But the latents now fall back to gradient descent (None = default = Adam). Pollux works out which blocks still admit a closed form rather than being told, and it warns when one does not.

## Latent Variable Models can have multiple outputs

So far each object has had one kind of data attached to it. LVMs are particularly useful when we have multiple types of data associated with each object. This is closely related to what machine learning calls "multi-modal learning," where a single representation has to account for several different kinds of measurement. This is also related to "multi-task learning," where a shared representation feeds several prediction targets. The difference here is that the latents are not computed from an input by an encoder --- they are free parameters we infer per object, which is what lets us apply a trained model to an object that has only one of its outputs. Thus, LVMs are more conceptually similar to "foundation models," but foundation models are often much more complex and less interpretable (e.g., contain neural network components).

In our case, LVMs are generative models and are connected to the data through a likelihood function. The likelihood can have multiple outputs, each with its own mapping from the latents to the predicted data. Each output can have its own dimensionality, units, and uncertainties. The latents are shared across all outputs, which allows the model to learn a common representation of the underlying structure in the data. For example, in the case of stellar spectroscopy, we might have a spectrum, a set of stellar labels from some pipeline, and broadband photometry for the same sources. 

Here is a graphical model of an LVM with two outputs, where $f$ represents the spectral flux and $\ell$ represents the stellar labels. The latents $z$ are shared across both outputs, and the coefficients $\theta_f$ and $\theta_\ell$ are specific to each output. Each output has its own likelihood function, which allows for different noise models and uncertainties for each type of data.

In [ ]:
pgm = daft.PGM(observed_style="shaded", dpi=150)
pgm.add_node("theta1", r"$\theta_f$", 1, 3)
pgm.add_node("theta2", r"$\theta_\ell$", 3, 3)
pgm.add_node("z", r"$z_n$", 2, 2)
pgm.add_node("f", r"$f_n$", 1, 1, observed=True, aspect=1.3)
pgm.add_node("l", r"$\ell_n$", 3, 1, observed=True, aspect=1.3)
for parent, child in [("z", "f"), ("z", "l"), ("theta1", "f"), ("theta2", "l")]:
    pgm.add_edge(parent, child)
pgm.add_plate(
    [0.4, 0.15, 3.2, 2.35],
    label=r"$n = 1 \ldots N$",
    shift=0.0,
    position="bottom right",
)
pgm.render();

The joint posterior probability factorizes the same way, with one likelihood term per output:

$$
p(\{z_n\}, \theta_f, \theta_\ell \mid \{f_n\}, \{\ell_n\}) \propto 
    p(\theta_f) \, p(\theta_\ell) \prod_{n=1}^{N} p(f_n \mid z_n, \theta_f) \, 
    p(\ell_n \mid z_n,\theta_\ell) \,
    p(z_n)
$$

The important structural point is that there is no term coupling $f_n$ to $\ell_n$. The spectrum and the labels are conditionally independent given $z_n$, so every bit of information one carries about the other travels through the shared latent vector.  This is also what makes the "train-then-apply" mechanism work with the model. For an object with only a spectrum, the label term is absent and the posterior on its latents is

$$
p(z_n \mid f_n, \theta_f) \propto p(f_n \mid z_n, \theta_f) \, p(z_n)
$$

We infer $z_n$ from the spectrum alone, then evaluate the label branch at that $z_n$ to predict $\ell_n$, using coefficients $\theta_\ell$ learned from the objects that did have labels. 

In Pollux, we can support multiple outputs using further calls to
{py:meth}`~pollux.models.LVM.register_output` to register new outputs and transformations from the latents. Each output gets its own transform, its own dimensionality, and its own component of the data and uncertainties, and they share the latents.

To demonstrate this, we will generate two "labels" per object from the same latents we used for the spectra, with much smaller uncertainties --- think of them as measurements from some external catalog that exists for only part of our sample:

In [ ]:
n_labels = 2

B = rng.normal(size=(n_labels, L))
label_err = np.full((N, n_labels), 0.05)
labels = z @ B.T + rng.normal(scale=label_err)

In [ ]:
data_multi = plx.data.PolluxData(
    flux=plx.data.OutputData(y, err=y_err),
    label=plx.data.OutputData(labels, err=label_err),
)

In [ ]:
model_multi = plx.LVM(latent_size=L)
model_multi.register_output("flux", LinearTransform(output_size=D))
model_multi.register_output("label", LinearTransform(output_size=n_labels))

One thing we can do with multiple outputs is to "train" a model on objects that have both outputs, and then apply the model to objects that only have one of the outputs. For example, we could train a model on objects that have both spectra and labels, and then apply the model to objects that only have spectra, to infer their labels. This is a powerful feature of LVMs, as it allows us to leverage all available data to learn a shared representation of the underlying structure in the data and then use this to make predictions for objects that have only a subset of the outputs.

In [ ]:
# train on half the objects, which have both kinds of data
train_data = data_multi[: N // 2]
trained_multi = model_multi.optimize_iterative(
    train_data, max_cycles=64, rng_key=jax.random.PRNGKey(0), progress=False
)

# the other half stand in for objects we only have a spectrum for
test_data = data_multi[N // 2 :]
test_flux_only = plx.data.PolluxData(flux=test_data["flux"])

applied = model_multi.optimize_iterative(
    test_flux_only,
    blocks=["latents"],
    fixed_pars=model_multi.output_pars(trained_multi.params),
    progress=False,
)
predicted_labels = model_multi.predict_outputs(applied.params, names="label")["label"]

In [ ]:
true_labels = np.asarray(test_data["label"].data)

fig, axes = plt.subplots(1, n_labels, figsize=(5 * n_labels, 4.5), layout="constrained")
for i, ax in enumerate(np.atleast_1d(axes)):
    ax.scatter(true_labels[:, i], predicted_labels[:, i], s=4, color="k", alpha=0.5)
    lims = np.percentile(true_labels[:, i], [1, 99])
    ax.axline((0, 0), slope=1, zorder=-10, color="tab:green", lw=1)
    rmse = np.sqrt(np.mean((predicted_labels[:, i] - true_labels[:, i]) ** 2))
    ax.set(xlabel=f"true label {i + 1}", ylabel=f"predicted label {i + 1}")
_ = fig.suptitle("labels predicted from the spectrum alone", fontsize=20)

## Where to go next

- [Getting Started](LVM-getting-started.ipynb) walks through the same model hands on,
  including the iterative optimizer and the preprocessing step we skipped here.
- [Linearized, closed-form solves](../linear-solves.md) covers how the fit exploits the
  bilinear structure discussed above.
- [Error models](LVM-error-models.ipynb) is about fitting an unknown intrinsic scatter $s$ for a data output.
- [Missing labels](LVM-hierarchical-missing-labels.ipynb) uses a trick to handle missing data.